# Transfer learning for image classification

**JobReady AI lab.** Run each cell with Shift+Enter. Try the exercises before opening the solutions at the end.

**You will:**
- Fine-tune a pretrained ResNet-18
- Use data augmentation
- Read a confusion matrix of mistakes

Use a GPU runtime: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch, torch.nn as nn, torchvision as tv
from torch.utils.data import DataLoader, Subset
device = 'cuda' if torch.cuda.is_available() else 'cpu'
norm = tv.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = tv.transforms.Compose([tv.transforms.Resize(128), tv.transforms.RandomHorizontalFlip(), tv.transforms.RandomResizedCrop(112, scale=(0.7, 1)), tv.transforms.ToTensor(), norm])
test_tf = tv.transforms.Compose([tv.transforms.Resize(128), tv.transforms.CenterCrop(112), tv.transforms.ToTensor(), norm])
train = Subset(tv.datasets.CIFAR10('data', train=True, download=True, transform=train_tf), range(5000))
test = Subset(tv.datasets.CIFAR10('data', train=False, download=True, transform=test_tf), range(2000))
train_dl = DataLoader(train, batch_size=64, shuffle=True, num_workers=2)
test_dl = DataLoader(test, batch_size=128, num_workers=2)

## 1. Load a pretrained network and replace the head

In [ ]:
model = tv.models.resnet18(weights='IMAGENET1K_V1')
for p in model.parameters(): p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)
opt = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)

## 2. Train the head, then evaluate

In [ ]:
def run_epoch():
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = nn.functional.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
def evaluate():
    model.eval(); preds, ys = [], []
    with torch.no_grad():
        for xb, yb in test_dl:
            preds.append(model(xb.to(device)).argmax(1).cpu()); ys.append(yb)
    return torch.cat(preds), torch.cat(ys)
for e in range(3):
    run_epoch(); p, t = evaluate(); print('epoch', e, 'accuracy', (p == t).float().mean().item())

## 3. Which classes get confused?

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print(classification_report(t, p, target_names=tv.datasets.CIFAR10.classes if hasattr(tv.datasets.CIFAR10, 'classes') else None))
print(confusion_matrix(t, p))

## Exercises
1. Unfreeze `model.layer4` and train 2 more epochs with lr = 1e-4. Does accuracy improve?
2. Remove augmentation and compare.

In [ ]:
# your work here

## Solutions

In [ ]:
for p_ in model.layer4.parameters(): p_.requires_grad = True
opt = torch.optim.AdamW([{'params': model.fc.parameters(), 'lr': 1e-3}, {'params': model.layer4.parameters(), 'lr': 1e-4}])
for e in range(2):
    run_epoch(); p, t = evaluate(); print('accuracy', (p == t).float().mean().item())